# Leakage-safe predictive vs predicted matrix

This notebook compares every available predictive/predicted pair using **expanding-window out-of-sample forecasts only**.

The key rule is: for a prediction at period `t`, the model is fitted only on periods strictly before `t`. The model never sees future target entries, future predictor entries, or same-period predictor values.

## What changed from the earlier version

The earlier notebook could look misleading because it plotted fitted training rows together with held-out rows, and the OLS model could be too flexible when there were only a few complete observations.

This version avoids that by:

- using **positive lags only**, so `LAGS = [1, 2, 3, 4]`, not `0`;
- using an **expanding-origin split**, so each row is predicted by a model trained only on previous rows;
- using **ridge regression** for each pair, which is less fragile than OLS with tiny samples;
- reporting best and worst held-out errors across all predictor/target pairs;
- computing a spike score for hospital-admission targets using only the training history available at the prediction date.

## Dataset roles

Automatically handled predictive series:

- Google Trends 1-year files in `Google_trends_v2/1y_data/time_series_GB*`
- UKHSA charts classified as NHS-call series
- raw wastewater files listed in `sources.csv` and stored under `data/raw/`
- processed wastewater long-format data, if `data/processed/wastewater_long.{parquet,csv}` also exists

Automatically handled predicted series:

- UKHSA charts classified as GP/admission series

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from wastewater.io import find_repo_root
from wastewater.regression_matrix import (
    build_available_series,
    summarise_available_series,
    expected_family_status,
)
from wastewater.leakage_safe_matrix import (
    run_leakage_safe_pairwise_matrix,
    best_worst_summary,
)

ROOT = find_repo_root(ROOT)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
ROOT

## 1. Build canonical series catalogue

Everything is converted into a simple long format with columns like `date`, `value`, `series_id`, `role`, and `dataset_family`. For raw wastewater, the notebook reads files in `data/raw` using `sources.csv`, infers date/value columns, and skips tables that cannot be parsed safely.

In [ ]:
series = build_available_series(ROOT)
print(f'Canonical observations: {len(series):,}')
display(series.head())

## 2. Check which expected families are available

In [ ]:
family_status = expected_family_status(ROOT, series)
display(family_status)

## 3. Inspect available series

In [ ]:
series_summary = summarise_available_series(series)
display(series_summary)

if not series_summary.empty:
    display(series_summary.groupby(['role', 'dataset_family'])['series_id'].nunique().reset_index(name='n_series'))

## 4. Run leakage-safe expanding-origin forecasts

For each predictive/predicted pair, the evaluation does this repeatedly:

1. aggregate both series to the chosen frequency;
2. create strictly positive predictor lags;
3. for each test period `t`, train on rows with period `< t`;
4. standardise using only that training window;
5. predict period `t`;
6. move forward and repeat.

This gives many genuine out-of-sample predictions rather than a fitted curve that has seen the future.

In [ ]:
FREQ = 'W'
LAGS = [1, 2, 3, 4]  # positive lags only; lag 0 is deliberately excluded
INITIAL_TRAIN_FRACTION = 0.6
MIN_TRAIN_SIZE = 12
RIDGE_ALPHA = 1.0
SPIKE_QUANTILE = 0.90

results, predictions = run_leakage_safe_pairwise_matrix(
    series,
    freq=FREQ,
    lags=LAGS,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    min_train_size=MIN_TRAIN_SIZE,
    ridge_alpha=RIDGE_ALPHA,
    spike_quantile=SPIKE_QUANTILE,
    aggregation='mean',
)

display(results)

results_path = PROCESSED / 'leakage_safe_pairwise_forecast_results.csv'
predictions_path = PROCESSED / 'leakage_safe_pairwise_forecast_predictions.csv'
results.to_csv(results_path, index=False)
predictions.to_csv(predictions_path, index=False)
print(f'Saved results to {results_path.relative_to(ROOT)}')
print(f'Saved predictions to {predictions_path.relative_to(ROOT)}')

## 5. Leakage checks

Each prediction row stores the prediction period and the last training period. The first check below should always be true: `train_end < period`.

In [ ]:
if predictions.empty:
    print('No successful leakage-safe predictions were produced.')
else:
    leakage_ok = (pd.to_datetime(predictions['train_end']) < pd.to_datetime(predictions['period'])).all()
    print('All predictions trained only on strictly earlier periods:', leakage_ok)
    display(predictions[['predictor_id', 'target_id', 'period', 'train_end', 'n_train', 'actual', 'prediction', 'abs_error']].head())

## 6. Best and worst held-out accuracy

This gives the worst-case and best-case pairings.

- `best_rmse`: smallest held-out RMSE.
- `worst_rmse`: largest held-out RMSE.
- `worst_abs_error`: largest single-period error.
- `best_skill`: best improvement over a rolling training-mean baseline.

In [ ]:
summaries = best_worst_summary(results)
summary_cols = [
    'predictor_family', 'predictor_name',
    'target_family', 'target_name',
    'n_complete', 'n_predictions',
    'rmse', 'baseline_rmse', 'mse_skill_vs_rolling_train_mean',
    'mae', 'max_abs_error', 'correlation', 'r2',
]

for label, table in summaries.items():
    print('\n' + label)
    if table.empty:
        print('No successful rows')
    else:
        display(table[[c for c in summary_cols if c in table.columns]])

## 7. Spike risk score for hospital admissions

For each prediction date, the spike threshold is the 90th percentile of the target series **in the training window available at that date**.

The spike score is an approximate probability, from 0 to 100, that the target will exceed that training-window spike threshold. It is calibrated from the model's training residual spread at that prediction date.

In [ ]:
ok = results[results['status'] == 'ok'].copy() if not results.empty and 'status' in results.columns else pd.DataFrame()

spike_cols = [
    'predictor_family', 'predictor_name',
    'target_family', 'target_name',
    'latest_prediction', 'latest_actual',
    'latest_spike_threshold', 'latest_spike_score', 'latest_actual_spike',
    'rmse', 'max_abs_error', 'mse_skill_vs_rolling_train_mean',
]

if ok.empty:
    print('No successful rows for spike scoring.')
else:
    spike_ranking = ok.sort_values('latest_spike_score', ascending=False)
    display(spike_ranking[[c for c in spike_cols if c in spike_ranking.columns]].head(30))
    spike_ranking.to_csv(PROCESSED / 'leakage_safe_hospital_spike_scores.csv', index=False)

## 8. Plot best and worst pairs using out-of-sample predictions only

No fitted training curve is shown here. The plotted predictions are only predictions generated by models that had not seen that period.

In [ ]:
def plot_pair(row, title_prefix):
    pred = predictions[
        (predictions['predictor_id'] == row['predictor_id']) &
        (predictions['target_id'] == row['target_id'])
    ].sort_values('period').copy()
    if pred.empty:
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(pred['period'], pred['actual'], label='Observed target', marker='o')
    ax.plot(pred['period'], pred['prediction'], label='Out-of-sample prediction', marker='o')
    ax.plot(pred['period'], pred['baseline_prediction'], label='Rolling train-mean baseline', linestyle='--')
    ax.set_title(title_prefix + ': ' + str(row['predictor_name']) + ' → ' + str(row['target_name']))
    ax.set_xlabel('Period')
    ax.set_ylabel('Target value')
    ax.legend()
    fig.autofmt_xdate()
    plt.show()
    display(pred[['period', 'train_end', 'n_train', 'actual', 'prediction', 'baseline_prediction', 'abs_error', 'spike_score']])

if not ok.empty and not predictions.empty:
    best = ok.sort_values('rmse', ascending=True).iloc[0]
    worst = ok.sort_values('rmse', ascending=False).iloc[0]
    worst_abs = ok.sort_values('max_abs_error', ascending=False).iloc[0]
    plot_pair(best, 'Best RMSE pair')
    plot_pair(worst, 'Worst RMSE pair')
    plot_pair(worst_abs, 'Worst single-period error pair')

## 9. Interpretation notes

- Treat the old train-curve visual fit as diagnostic only; the score here comes only from expanding-window out-of-sample predictions.
- If a pair has very few out-of-sample predictions, increase the available history or lower `MIN_TRAIN_SIZE` cautiously.
- Negative `mse_skill_vs_rolling_train_mean` means the pair is worse than simply predicting the rolling training mean.
- The worst-case tables are important: a predictor can look good on average while still producing unacceptable single-period misses.
- The spike score is a screening signal, not a calibrated clinical alert until validated on more seasons and regions.